# 제주도 전력 수요 예측 모델 (BiLSTM)
## Full Pipeline: 학습, 검증, 테스트 및 시각화

**목표 성능**:
- MAPE: 5-6%
- R²: ≥ 0.75

**모델 아키텍처**:
- BiLSTM (양방향 LSTM)
- Hidden Size: 64
- Layers: 2
- Dropout: 0.2

---

## 1. 환경 설정

In [ ]:
# Google Colab에서 실행 시 GitHub 레포지토리 클론
import os

if 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython()):
    print("Running on Google Colab")
    !git clone https://github.com/kiminbean/power-demand-forecast.git
    %cd power-demand-forecast
    !pip install -q torch pandas numpy scikit-learn matplotlib seaborn
else:
    print("Running locally")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Device 설정
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. 데이터 로딩 및 탐색 (EDA)

In [ ]:
# 데이터 로드
data_path = 'data/processed/jeju_hourly_merged.csv'
df = pd.read_csv(data_path, encoding='utf-8-sig')

# datetime 파싱
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print(f"데이터 형태: {df.shape}")
print(f"기간: {df['datetime'].min()} ~ {df['datetime'].max()}")
print(f"\n컬럼 목록:")
print(df.columns.tolist())

In [ ]:
# 기본 통계
print("=" * 60)
print("전력 수요 (power_demand) 통계")
print("=" * 60)
print(df['power_demand'].describe())
print(f"\n결측치: {df['power_demand'].isna().sum()}")

In [ ]:
# 전력 수요 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램
axes[0].hist(df['power_demand'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Power Demand (MW)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Power Demand Distribution')
axes[0].axvline(df['power_demand'].mean(), color='red', linestyle='--', label=f'Mean: {df["power_demand"].mean():.1f}')

# 박스플롯
axes[1].boxplot(df['power_demand'].dropna())
axes[1].set_ylabel('Power Demand (MW)')
axes[1].set_title('Power Demand Boxplot')

axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 시계열 플롯
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# 전체 기간
axes[0].plot(df['datetime'], df['power_demand'], linewidth=0.5, alpha=0.7)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Power Demand (MW)')
axes[0].set_title('Power Demand Time Series (2013-2024)')
axes[0].grid(True, alpha=0.3)

# 최근 1개월
recent = df[df['datetime'] >= df['datetime'].max() - timedelta(days=30)]
axes[1].plot(recent['datetime'], recent['power_demand'], linewidth=1)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Power Demand (MW)')
axes[1].set_title('Power Demand - Last 30 Days')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 시간대별 패턴
df['hour'] = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['year'] = df['datetime'].dt.year

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 시간대별
hourly = df.groupby('hour')['power_demand'].mean()
axes[0, 0].bar(hourly.index, hourly.values, color='steelblue')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Average Power Demand (MW)')
axes[0, 0].set_title('Average Power Demand by Hour')
axes[0, 0].set_xticks(range(0, 24))

# 요일별
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = df.groupby('dayofweek')['power_demand'].mean()
axes[0, 1].bar(range(7), daily.values, color='coral')
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Average Power Demand (MW)')
axes[0, 1].set_title('Average Power Demand by Day of Week')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(days)

# 월별
monthly = df.groupby('month')['power_demand'].mean()
axes[1, 0].bar(monthly.index, monthly.values, color='forestgreen')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Average Power Demand (MW)')
axes[1, 0].set_title('Average Power Demand by Month')
axes[1, 0].set_xticks(range(1, 13))

# 연도별 추이
yearly = df.groupby('year')['power_demand'].mean()
axes[1, 1].plot(yearly.index, yearly.values, marker='o', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Average Power Demand (MW)')
axes[1, 1].set_title('Average Power Demand by Year')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 피처 엔지니어링 (38개 피처)

In [ ]:
def create_features(df):
    """38개 피처 생성"""
    data = df.copy()
    
    # 1. 시간 피처 (8개)
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    data['dayofweek_sin'] = np.sin(2 * np.pi * data['dayofweek'] / 7)
    data['dayofweek_cos'] = np.cos(2 * np.pi * data['dayofweek'] / 7)
    data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
    data['is_weekend'] = (data['dayofweek'] >= 5).astype(int)
    
    # 공휴일 (간단 버전 - 실제로는 holidays 라이브러리 사용)
    data['is_holiday'] = 0
    
    # 2. 기상 피처 (주요 변수들)
    # 기온 관련
    if '기온' in data.columns:
        data['temp'] = pd.to_numeric(data['기온'], errors='coerce')
    else:
        data['temp'] = 15  # 기본값
    
    if '습도' in data.columns:
        data['humidity'] = pd.to_numeric(data['습도'], errors='coerce')
    else:
        data['humidity'] = 60
    
    if '풍속' in data.columns:
        data['wind_speed'] = pd.to_numeric(data['풍속'], errors='coerce')
    else:
        data['wind_speed'] = 3
    
    # THI (불쾌지수)
    data['THI'] = 1.8 * data['temp'] - 0.55 * (1 - data['humidity']/100) * (1.8 * data['temp'] - 26) + 32
    
    # HDD/CDD (난방/냉방 도일)
    data['HDD'] = np.maximum(18 - data['temp'], 0)
    data['CDD'] = np.maximum(data['temp'] - 18, 0)
    
    # 3. 래그 피처 (7개)
    data['demand_lag_1h'] = data['power_demand'].shift(1)
    data['demand_lag_24h'] = data['power_demand'].shift(24)
    data['demand_lag_168h'] = data['power_demand'].shift(168)  # 1주일
    data['demand_ma_24h'] = data['power_demand'].rolling(24).mean().shift(1)
    data['demand_ma_168h'] = data['power_demand'].rolling(168).mean().shift(1)
    data['temp_lag_1h'] = data['temp'].shift(1)
    data['temp_lag_24h'] = data['temp'].shift(24)
    
    # 4. 일사/태양광 관련
    if '일사' in data.columns:
        data['irradiance'] = pd.to_numeric(data['일사'], errors='coerce').fillna(0)
    else:
        data['irradiance'] = 0
    
    # 주간 플래그
    data['daylight'] = ((data['hour'] >= 6) & (data['hour'] <= 18)).astype(int)
    
    # 5. 외부 피처
    if 'ev_cumulative' in data.columns:
        data['ev_cumsum'] = pd.to_numeric(data['ev_cumulative'], errors='coerce').fillna(method='ffill')
    else:
        data['ev_cumsum'] = 0
    
    if 'visitors' in data.columns:
        data['visitors_daily'] = pd.to_numeric(data['visitors'], errors='coerce').fillna(method='ffill')
    else:
        data['visitors_daily'] = 0
    
    return data

# 피처 생성
df_features = create_features(df)
print(f"피처 생성 완료: {df_features.shape}")

In [ ]:
# 사용할 피처 목록 정의
feature_cols = [
    # 시간 피처
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos',
    'month_sin', 'month_cos', 'is_weekend', 'is_holiday',
    # 기상 피처
    'temp', 'humidity', 'wind_speed', 'THI', 'HDD', 'CDD',
    # 래그 피처
    'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
    'demand_ma_24h', 'demand_ma_168h', 'temp_lag_1h', 'temp_lag_24h',
    # 기타
    'irradiance', 'daylight', 'ev_cumsum', 'visitors_daily'
]

# 실제 존재하는 피처만 선택
feature_cols = [col for col in feature_cols if col in df_features.columns]
print(f"사용할 피처 수: {len(feature_cols)}")
print(f"피처 목록: {feature_cols}")

In [ ]:
# 결측치 처리
df_clean = df_features.dropna(subset=['power_demand'] + feature_cols)
print(f"결측치 제거 후: {df_clean.shape[0]} / {df_features.shape[0]} ({100*df_clean.shape[0]/df_features.shape[0]:.1f}%)")

## 4. 데이터 분할 (Train / Validation / Test)

In [ ]:
# 시간 기반 분할 (데이터 누수 방지)
# Train: 70%, Validation: 15%, Test: 15%

n = len(df_clean)
train_size = int(n * 0.70)
val_size = int(n * 0.15)

train_df = df_clean.iloc[:train_size]
val_df = df_clean.iloc[train_size:train_size+val_size]
test_df = df_clean.iloc[train_size+val_size:]

print(f"Train: {len(train_df):,} samples ({train_df['datetime'].min()} ~ {train_df['datetime'].max()})")
print(f"Val:   {len(val_df):,} samples ({val_df['datetime'].min()} ~ {val_df['datetime'].max()})")
print(f"Test:  {len(test_df):,} samples ({test_df['datetime'].min()} ~ {test_df['datetime'].max()})")

In [ ]:
# 스케일링 (Train 데이터에서만 fit - 데이터 누수 방지)
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit on training data only
X_train = feature_scaler.fit_transform(train_df[feature_cols])
y_train = target_scaler.fit_transform(train_df[['power_demand']])

# Transform validation and test
X_val = feature_scaler.transform(val_df[feature_cols])
y_val = target_scaler.transform(val_df[['power_demand']])

X_test = feature_scaler.transform(test_df[feature_cols])
y_test = target_scaler.transform(test_df[['power_demand']])

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

## 5. 시퀀스 데이터셋 생성

In [ ]:
class TimeSeriesDataset(Dataset):
    """시계열 데이터셋"""
    def __init__(self, X, y, seq_length=48):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.seq_length = seq_length
        
    def __len__(self):
        return len(self.X) - self.seq_length
    
    def __getitem__(self, idx):
        return (
            self.X[idx:idx+self.seq_length],
            self.y[idx+self.seq_length]
        )

# 하이퍼파라미터
SEQ_LENGTH = 48  # 48시간 (2일)
BATCH_SIZE = 32

# 데이터셋 생성
train_dataset = TimeSeriesDataset(X_train, y_train, SEQ_LENGTH)
val_dataset = TimeSeriesDataset(X_val, y_val, SEQ_LENGTH)
test_dataset = TimeSeriesDataset(X_test, y_test, SEQ_LENGTH)

# 데이터로더
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 6. BiLSTM 모델 정의

In [ ]:
class BiLSTMModel(nn.Module):
    """양방향 LSTM 모델"""
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        
        # BiLSTM이므로 hidden_size * 2
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )
        
        self._init_weights()
        
    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
                n = param.size(0)
                param.data[n//4:n//2].fill_(1)
        
        for module in self.fc:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
                if module.bias is not None:
                    module.bias.data.fill_(0)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # 마지막 타임스텝
        last_output = lstm_out[:, -1, :]
        return self.fc(last_output)

# 모델 생성
INPUT_SIZE = len(feature_cols)
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2

model = BiLSTMModel(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# 파라미터 수 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Architecture:")
print(f"  Input Size: {INPUT_SIZE}")
print(f"  Hidden Size: {HIDDEN_SIZE}")
print(f"  Num Layers: {NUM_LAYERS}")
print(f"  Dropout: {DROPOUT}")
print(f"  Total Parameters: {total_params:,}")
print(f"  Trainable Parameters: {trainable_params:,}")

## 7. 학습

In [ ]:
# 학습 설정
EPOCHS = 50
LEARNING_RATE = 0.001
PATIENCE = 10  # Early stopping

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

print(f"Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Early Stopping Patience: {PATIENCE}")

In [ ]:
# 학습 루프
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print("Starting training...")
print("=" * 60)

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            loss = criterion(output, y_batch)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        marker = " ✓ Best"
    else:
        patience_counter += 1
        marker = ""
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:2d}/{EPOCHS}] Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}{marker}")
    
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Best 모델 복원
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("=" * 60)
print(f"Training completed. Best Val Loss: {best_val_loss:.6f}")

In [ ]:
# 학습 곡선
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Learning Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss (log scale)')
plt.title('Learning Curves (Log Scale)')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 평가

In [ ]:
def evaluate_model(model, data_loader, target_scaler, device):
    """모델 평가 및 예측값 반환"""
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            output = model(X_batch)
            predictions.extend(output.cpu().numpy().flatten())
            actuals.extend(y_batch.numpy().flatten())
    
    # 역스케일링
    predictions = target_scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
    actuals = target_scaler.inverse_transform(np.array(actuals).reshape(-1, 1)).flatten()
    
    # 메트릭 계산
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    
    return {
        'predictions': predictions,
        'actuals': actuals,
        'mape': mape,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

In [ ]:
# 각 데이터셋 평가
train_results = evaluate_model(model, train_loader, target_scaler, device)
val_results = evaluate_model(model, val_loader, target_scaler, device)
test_results = evaluate_model(model, test_loader, target_scaler, device)

print("=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)
print(f"\n{'Dataset':<10} {'MAPE (%)':>10} {'RMSE (MW)':>12} {'MAE (MW)':>10} {'R²':>8}")
print("-" * 60)
print(f"{'Train':<10} {train_results['mape']:>10.2f} {train_results['rmse']:>12.2f} {train_results['mae']:>10.2f} {train_results['r2']:>8.4f}")
print(f"{'Validation':<10} {val_results['mape']:>10.2f} {val_results['rmse']:>12.2f} {val_results['mae']:>10.2f} {val_results['r2']:>8.4f}")
print(f"{'Test':<10} {test_results['mape']:>10.2f} {test_results['rmse']:>12.2f} {test_results['mae']:>10.2f} {test_results['r2']:>8.4f}")
print("=" * 60)

In [ ]:
# 목표 달성 여부 확인
print("\n" + "=" * 60)
print("TARGET ACHIEVEMENT")
print("=" * 60)

mape_target = 6.0
r2_target = 0.75

mape_achieved = test_results['mape'] <= mape_target
r2_achieved = test_results['r2'] >= r2_target

print(f"\nMAPE Target: ≤ {mape_target}% | Achieved: {test_results['mape']:.2f}% {'✅' if mape_achieved else '❌'}")
print(f"R² Target: ≥ {r2_target} | Achieved: {test_results['r2']:.4f} {'✅' if r2_achieved else '❌'}")
print("=" * 60)

## 9. 시각화

In [ ]:
# 예측 vs 실제 (Test 데이터)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 시계열 비교 (전체)
axes[0, 0].plot(test_results['actuals'], label='Actual', alpha=0.7, linewidth=0.8)
axes[0, 0].plot(test_results['predictions'], label='Predicted', alpha=0.7, linewidth=0.8)
axes[0, 0].set_xlabel('Time Step')
axes[0, 0].set_ylabel('Power Demand (MW)')
axes[0, 0].set_title('Prediction vs Actual (Full Test Period)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. 시계열 비교 (최근 7일)
n_recent = 24 * 7  # 7일
axes[0, 1].plot(test_results['actuals'][-n_recent:], label='Actual', linewidth=1.5)
axes[0, 1].plot(test_results['predictions'][-n_recent:], label='Predicted', linewidth=1.5)
axes[0, 1].set_xlabel('Time Step')
axes[0, 1].set_ylabel('Power Demand (MW)')
axes[0, 1].set_title('Prediction vs Actual (Last 7 Days)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. 산점도
axes[1, 0].scatter(test_results['actuals'], test_results['predictions'], alpha=0.3, s=10)
min_val = min(test_results['actuals'].min(), test_results['predictions'].min())
max_val = max(test_results['actuals'].max(), test_results['predictions'].max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1, 0].set_xlabel('Actual Power Demand (MW)')
axes[1, 0].set_ylabel('Predicted Power Demand (MW)')
axes[1, 0].set_title(f'Scatter Plot (R² = {test_results["r2"]:.4f})')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 오차 분포
errors = test_results['actuals'] - test_results['predictions']
axes[1, 1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Prediction Error (MW)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title(f'Error Distribution (Mean: {np.mean(errors):.2f}, Std: {np.std(errors):.2f})')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 시간대별 오차 분석
test_df_eval = test_df.iloc[SEQ_LENGTH:].copy().reset_index(drop=True)
test_df_eval = test_df_eval.iloc[:len(test_results['predictions'])].copy()
test_df_eval['prediction'] = test_results['predictions']
test_df_eval['actual'] = test_results['actuals']
test_df_eval['error'] = np.abs(test_df_eval['actual'] - test_df_eval['prediction'])
test_df_eval['pct_error'] = test_df_eval['error'] / test_df_eval['actual'] * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 시간대별 MAPE
hourly_mape = test_df_eval.groupby('hour')['pct_error'].mean()
axes[0, 0].bar(hourly_mape.index, hourly_mape.values, color='steelblue')
axes[0, 0].axhline(test_results['mape'], color='red', linestyle='--', label=f'Overall MAPE: {test_results["mape"]:.2f}%')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('MAPE (%)')
axes[0, 0].set_title('MAPE by Hour of Day')
axes[0, 0].set_xticks(range(0, 24))
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 요일별 MAPE
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_mape = test_df_eval.groupby('dayofweek')['pct_error'].mean()
axes[0, 1].bar(range(7), daily_mape.values, color='coral')
axes[0, 1].axhline(test_results['mape'], color='red', linestyle='--', label=f'Overall MAPE: {test_results["mape"]:.2f}%')
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('MAPE (%)')
axes[0, 1].set_title('MAPE by Day of Week')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(days)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 월별 MAPE
monthly_mape = test_df_eval.groupby('month')['pct_error'].mean()
axes[1, 0].bar(monthly_mape.index, monthly_mape.values, color='forestgreen')
axes[1, 0].axhline(test_results['mape'], color='red', linestyle='--', label=f'Overall MAPE: {test_results["mape"]:.2f}%')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('MAPE (%)')
axes[1, 0].set_title('MAPE by Month')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 주중 vs 주말
weekend_mape = test_df_eval.groupby('is_weekend')['pct_error'].mean()
axes[1, 1].bar(['Weekday', 'Weekend'], weekend_mape.values, color=['steelblue', 'coral'])
axes[1, 1].axhline(test_results['mape'], color='red', linestyle='--', label=f'Overall MAPE: {test_results["mape"]:.2f}%')
axes[1, 1].set_ylabel('MAPE (%)')
axes[1, 1].set_title('MAPE: Weekday vs Weekend')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 잔차 분석
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

errors = test_results['actuals'] - test_results['predictions']

# Q-Q 플롯 (정규성 검정)
from scipy import stats
stats.probplot(errors, dist="norm", plot=axes[0, 0])
axes[0, 0].set_title('Q-Q Plot (Residuals)')

# 잔차 vs 예측값
axes[0, 1].scatter(test_results['predictions'], errors, alpha=0.3, s=10)
axes[0, 1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted Value (MW)')
axes[0, 1].set_ylabel('Residual (MW)')
axes[0, 1].set_title('Residuals vs Predicted Values')
axes[0, 1].grid(True, alpha=0.3)

# 잔차 시계열
axes[1, 0].plot(errors, alpha=0.7, linewidth=0.5)
axes[1, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Time Step')
axes[1, 0].set_ylabel('Residual (MW)')
axes[1, 0].set_title('Residuals Over Time')
axes[1, 0].grid(True, alpha=0.3)

# 자기상관 (ACF)
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(pd.Series(errors), ax=axes[1, 1])
axes[1, 1].set_title('Autocorrelation of Residuals')

plt.tight_layout()
plt.show()

## 10. 신뢰 구간

In [ ]:
# 오차의 표준편차를 이용한 신뢰구간 계산
std_error = np.std(errors)

# 90% 신뢰구간 (1.645 * std)
confidence_90 = 1.645 * std_error

# 최근 48시간 예측 + 신뢰구간
n_plot = 48
recent_pred = test_results['predictions'][-n_plot:]
recent_actual = test_results['actuals'][-n_plot:]

plt.figure(figsize=(14, 6))
x = range(n_plot)

plt.fill_between(x, recent_pred - confidence_90, recent_pred + confidence_90, 
                 alpha=0.3, color='blue', label='90% Confidence Interval')
plt.plot(x, recent_actual, 'k-', linewidth=2, label='Actual')
plt.plot(x, recent_pred, 'b--', linewidth=2, label='Predicted')

plt.xlabel('Time Step')
plt.ylabel('Power Demand (MW)')
plt.title('Prediction with 90% Confidence Interval (Last 48 Hours)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 신뢰구간 커버리지 계산
in_interval = np.sum(np.abs(errors) <= confidence_90) / len(errors) * 100
print(f"90% Confidence Interval Coverage: {in_interval:.1f}%")

## 11. 모델 저장

In [ ]:
# 모델 저장
import os

save_dir = 'models/bilstm_demand'
os.makedirs(save_dir, exist_ok=True)

# 모델 가중치 저장
torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': INPUT_SIZE,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'seq_length': SEQ_LENGTH,
    'feature_cols': feature_cols,
    'metrics': {
        'train_mape': train_results['mape'],
        'val_mape': val_results['mape'],
        'test_mape': test_results['mape'],
        'test_r2': test_results['r2'],
        'test_rmse': test_results['rmse'],
        'test_mae': test_results['mae']
    }
}, f'{save_dir}/bilstm_model.pt')

# 스케일러 저장
import pickle
with open(f'{save_dir}/feature_scaler.pkl', 'wb') as f:
    pickle.dump(feature_scaler, f)
with open(f'{save_dir}/target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

print(f"Model saved to {save_dir}/")
print(f"Files: bilstm_model.pt, feature_scaler.pkl, target_scaler.pkl")

## 12. 결과 요약

In [ ]:
print("=" * 70)
print("               POWER DEMAND FORECAST MODEL SUMMARY")
print("=" * 70)
print(f"\n📊 MODEL ARCHITECTURE")
print(f"   Type: BiLSTM (Bidirectional LSTM)")
print(f"   Input Size: {INPUT_SIZE} features")
print(f"   Hidden Size: {HIDDEN_SIZE}")
print(f"   Num Layers: {NUM_LAYERS}")
print(f"   Dropout: {DROPOUT}")
print(f"   Sequence Length: {SEQ_LENGTH} hours")
print(f"   Parameters: {total_params:,}")

print(f"\n📈 PERFORMANCE METRICS")
print(f"   {'Dataset':<12} {'MAPE':>8} {'R²':>8} {'RMSE':>10} {'MAE':>10}")
print(f"   {'-'*50}")
print(f"   {'Train':<12} {train_results['mape']:>7.2f}% {train_results['r2']:>8.4f} {train_results['rmse']:>9.2f}MW {train_results['mae']:>9.2f}MW")
print(f"   {'Validation':<12} {val_results['mape']:>7.2f}% {val_results['r2']:>8.4f} {val_results['rmse']:>9.2f}MW {val_results['mae']:>9.2f}MW")
print(f"   {'Test':<12} {test_results['mape']:>7.2f}% {test_results['r2']:>8.4f} {test_results['rmse']:>9.2f}MW {test_results['mae']:>9.2f}MW")

print(f"\n🎯 TARGET ACHIEVEMENT")
print(f"   MAPE ≤ 6%: {test_results['mape']:.2f}% {'✅ ACHIEVED' if test_results['mape'] <= 6.0 else '❌ NOT MET'}")
print(f"   R² ≥ 0.75: {test_results['r2']:.4f} {'✅ ACHIEVED' if test_results['r2'] >= 0.75 else '❌ NOT MET'}")

print(f"\n📅 DATA SPLIT")
print(f"   Train: {len(train_df):,} samples")
print(f"   Validation: {len(val_df):,} samples")
print(f"   Test: {len(test_df):,} samples")

print("\n" + "=" * 70)